# Blue Team Lab (Endpoint-centric)
## 60-minute investigation using endpoint telemetry

### Scenario
You are a blue team analyst reviewing endpoint telemetry: process creation, network connections, file writes, and persistence.

### Goals
- Identify the suspicious execution chain
- Identify the artifact and persistence mechanism
- Identify likely C2 destination
- Recommend containment and one detection improvement
- Document your findings clearly

### Deliverables
1) Answers to the 8 investigation questions
2) A 6–10 sentence incident summary


## Participant Guide

### What you are doing
You are investigating possible malicious activity using endpoint logs. Endpoint logs tell you what programs ran, what files were created, what network connections happened, and whether anything tried to stay on the computer after reboot.

### How to run this notebook
1) In Colab: Runtime → Run all.
2) Read the text above each code cell. It tells you what the cell does and what you should see.
3) When you reach the HOST timeline section, you will fill in `HOST = "..."` using a value you copy from the earlier output.

### What a good result looks like
- You can name the host that looks compromised.
- You can describe the suspicious process chain (who launched what).
- You can point to an artifact (file path, optional hash).
- You can point to a suspicious outbound destination.
- You can recommend containment and a detection improvement.


## Section 1: Load the data

### What
This cell loads the endpoint log data from a CSV file hosted in GitHub.

### Why
Colab does not automatically have files from your computer. Loading from a URL ensures everyone is using the same dataset.

### What to expect
- A small table preview (`df.head(10)`) with columns like `host`, `user`, `event_type`, `process`, `parent_process`, `cmdline`.

### If something goes wrong
- If you get an error here, stop. Nothing else will work until the data loads.


In [ ]:
import pandas as pd
import numpy as np

# Colab data source (raw GitHub URL)
ENDPOINT_CSV_URL = "https://raw.githubusercontent.com/dadirad/blue-team-lab/main/files/endpoint_events.csv"

df = pd.read_csv(ENDPOINT_CSV_URL)

# Normalize timestamps if present
if "ts" in df.columns:
    df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
    df = df.sort_values("ts").reset_index(drop=True)

df.head(10)


## Task 0: Confirm what data you have

### What
This cell counts the types of endpoint events.

### Why
It tells you what evidence exists in this dataset and what sections will be useful.

### What to expect
You should see event types like:
- PROCESS_CREATE (program starts)
- NETWORK_CONNECT (outbound connections)
- FILE_WRITE (files written to disk)
- REG_PERSIST or similar (persistence)


In [ ]:
df["event_type"].value_counts(dropna=False)


## Task 1: Identify suspicious parent → child process chains

### What
This cell looks for suspicious process relationships, like an email or Office app launching a scripting tool.

### Why
A very common attack path is:
1. user opens email or document
2. that app launches PowerShell or another script engine
3. the script downloads or runs a payload

### What to expect
You should see a table that includes:
- `host`
- `parent_process`
- `process`
- `cmdline`

### What to do
Pick the host with the strongest suspicious chain. You will use that host value later.


In [ ]:
proc = df[df["event_type"]=="PROCESS_CREATE"].copy()
proc["process_l"] = proc["process"].fillna("").str.lower()
proc["parent_l"] = proc["parent_process"].fillna("").str.lower()
proc["cmd_l"] = proc["cmdline"].fillna("").str.lower()

susp_chain = proc[
    (proc["parent_l"].str.contains("outlook|winword|excel|chrome|acrord32")) &
    (proc["process_l"].str.contains("powershell|cmd|wscript|cscript|mshta|rundll32"))
].copy()

susp_chain[["ts","host","user","parent_process","process","cmdline","notes"]].sort_values("ts") if "ts" in df.columns else susp_chain[["host","user","parent_process","process","cmdline","notes"]]


## Task 2: Look for encoded PowerShell or suspicious flags

### What
This cell filters PowerShell activity for command line patterns often used by attackers.

### Why
Attackers use flags like these to hide activity:
- `-enc` or `-EncodedCommand` (encoded script)
- `-nop` (no profile)
- hidden window settings

### What to expect
A smaller table of the most suspicious PowerShell entries.

### What to do
Capture one row as evidence (timestamp, host, command line pattern).


In [ ]:
ps = proc[proc["process_l"].eq("powershell.exe")].copy()
encoded_or_hidden = ps[
    ps["cmd_l"].str.contains(r" -enc | -encodedcommand | -nop | -w hidden | -windowstyle hidden ", regex=True)
].copy()

encoded_or_hidden[["ts","host","user","parent_process","process","cmdline","notes"]].sort_values("ts") if "ts" in df.columns else encoded_or_hidden[["host","user","parent_process","process","cmdline","notes"]]


## Task 3: Pivot to a single host timeline

### What
This section shows all events for one host in time order.

### Why
A timeline is how analysts confirm the story. One alert is not enough. A sequence of events is much stronger evidence.

### What to do
1) Copy a host value from Task 1 output (example: WS-023)
2) Paste it into `HOST = "..."`
3) Run the code cell again

### What to expect
A chain like:
- suspicious process starts
- outbound network connect
- file write (payload drop)
- payload execution
- persistence
- repeated network connections


In [ ]:
HOST = ""  # e.g., "WS-023"
if not HOST:
    print("Set HOST to a value like 'WS-023' and re-run this cell.")
else:
    focus = df[df["host"]==HOST].copy()
    cols = ["ts","event_type","host","user","process","parent_process","cmdline","hash","network_dst","network_port","action","notes"]
    cols = [c for c in cols if c in df.columns]
    display(focus[cols].sort_values("ts") if "ts" in df.columns else focus[cols])


## Task 4: Identify artifacts (file writes) and persistence

### What
This section lists:
- files written to disk (possible payloads)
- persistence events (how something tries to survive reboot)

### Why
File writes and persistence are strong signals that activity is not normal.

### What to expect
- FILE_WRITE rows with a file path (often Temp or user profile)
- PERSIST rows that reference a registry run key or similar

### What to do
Capture:
- artifact path
- hash (if present)
- persistence mechanism


In [ ]:
file_writes = df[df["event_type"]=="FILE_WRITE"].copy()
persist = df[df["event_type"].str.contains("PERSIST", na=False)].copy()

display(file_writes[[c for c in ["ts","host","user","process","cmdline","hash","notes"] if c in df.columns]].sort_values("ts") if "ts" in df.columns else file_writes[[c for c in ["host","user","process","cmdline","hash","notes"] if c in df.columns]])
display(persist[[c for c in ["ts","host","user","event_type","process","cmdline","notes"] if c in df.columns]].sort_values("ts") if "ts" in df.columns else persist[[c for c in ["host","user","event_type","process","cmdline","notes"] if c in df.columns]])


## Task 5: Identify likely C2 destination and beaconing

### What
This section summarizes outbound destinations and checks for repeated connections.

### Why
After running code, malware often connects back to an attacker server. Repeated connections at similar intervals can indicate beaconing.

### What to expect
A table showing destinations with counts and time gaps.

### What to do
Pick the most suspicious destination and note:
- destination domain or IP
- port
- repetition pattern


In [ ]:
net = df[df["event_type"]=="NETWORK_CONNECT"].copy()
if "ts" in df.columns and all(c in df.columns for c in ["host","process","network_dst","network_port"]):
    net = net.sort_values(["host","process","network_dst","network_port","ts"])
    net["delta_sec"] = net.groupby(["host","process","network_dst","network_port"])["ts"].diff().dt.total_seconds()
    net_stats = (
        net.groupby(["host","process","network_dst","network_port"], dropna=False)
           .agg(
               count=("ts","count"),
               mean_delta=("delta_sec","mean"),
               std_delta=("delta_sec","std"),
               first=("ts","min"),
               last=("ts","max")
           )
           .reset_index()
           .sort_values(["count","std_delta"], ascending=[False, True])
    )
    display(net_stats.head(25))
else:
    print("Beaconing stats require host, process, network_dst, network_port, and ts columns.")


## Submit: 8 Investigation Questions

### What
This is where you document your findings.

### Why
In real incident response, the write-up is how others take action: containment, remediation, and detection updates.

### What to do
Answer each question with evidence (timestamps, host, process chain, artifact, destination).

1) Suspected compromised host:
2) Suspicious process chain (parent → child) OR suspicious artifact:
3) First suspicious timestamp (with evidence):
4) Primary lead (what triggered your investigation?):
5) Evidence that supports malicious activity (2–3 bullets):
6) Best-guess attack technique (plain language is fine):
7) Immediate containment actions (1–2 actions):
8) One detection improvement idea:

## Incident Summary (6–10 sentences)
**Title:** Suspected endpoint compromise on `<host>`

- At `<time>`, `<parent process>` spawned `<child process>` with suspicious command-line behavior.
- The activity indicates `<download/execution>` of `<artifact>` (include hash if present).
- Persistence was established via `<mechanism>`.
- The host initiated repeated outbound connections to `<destination>` suggesting possible command and control.
- Confidence: `<high/medium/low>` because `<reason>`.
- Recommended containment: `<actions>`.
- Detection improvement: `<specific detection rule/tuning idea>`.
